# 06 — Full fine-tune (ceiling reference)

Update every parameter in MedSAM2 with a low learning rate. This is the
compute-heavy ceiling the PEFT runs are measured against. On a single
24 GB GPU this needs small batch + gradient checkpointing for the real
(larger) datasets; for the synthetic toy data it's fast.

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path
from cfrp_medsam2.train import TrainConfig, train
from cfrp_medsam2.model import ModelConfig

REPO = Path('..').resolve()

cfg = TrainConfig(
    regime='full_ft',
    model=ModelConfig(
        backend='medsam2',
        checkpoint=str(REPO / 'checkpoints' / 'sam2.1_hiera_tiny.pt'),
        image_size=512,
    ),
    train_volumes=tuple(str(p) for p in sorted((REPO / 'data' / 'processed' / 'toy').glob('train_*.npz'))),
    val_volumes=tuple(str(p) for p in sorted((REPO / 'data' / 'processed' / 'toy').glob('val_*.npz'))),
    image_size=512,
    batch_size=1,
    epochs=3,
    full_ft_lr=1.0e-5,
    prompt_mode='bbox',
    ckpt_dir=str(REPO / 'checkpoints'),
    log_path=str(REPO / 'logs' / 'full_ft.csv'),
)
result = train(cfg)
print('best_val_dice =', result['best_val_dice'])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv(REPO / 'logs' / 'full_ft.csv')
df.plot(x='epoch', y=['train_dice', 'val_dice'], figsize=(6, 3.5))
plt.title('Full fine-tune training curves')
plt.grid(True, alpha=0.3)